# 端到端流程验证：单张 CubiCasa 图纸

**目标**：用一张有 GT 标注的 CubiCasa 图纸，把六层流水线从头到尾跑一遍，每一步都打印中间结果，确认没有断点。

```
Cell 0  环境 & 路径
Cell 1  选图 & 加载 GT          ← 数据层验证
Cell 2  自适应预处理             ← utils.py
Cell 3  模型推理                 ← model_arch.py + post_service.py
Cell 4  分割质量验证 (IoU mask)  ← 第一个数字指标
Cell 5  矢量化                   ← vector_logic.py
Cell 6  矢量化质量验证 (IoU vect)
Cell 7  门窗检测质量 (F1)
Cell 8  3D 重建                  ← reconstruct_3d.py
Cell 9  3D 几何合理性检查
Cell 10 Staging + 审核模拟       ← preview/persistence_service.py
Cell 11 全流程指标汇总
```

跑完后所有 `✓` 亮起，说明流水线没有断点，可以开始工程化。

## Cell 0 · 环境 & 路径

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, sys, json, time, io
from pathlib import Path
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 路径（自动适配 Vast.ai / Colab / 本地）──
if os.path.exists('/workspace/production_3d'):
    PROJECT_ROOT  = '/workspace/production_3d'
    CUBICASA_ROOT = '/workspace/CubiCasa5k'
    DATA_FOLDER   = '/workspace/data/cubicasa5k/'
    CKPT_DIR      = '/workspace/production_3d/checkpoints_paper'
elif os.path.exists('/content'):
    PROJECT_ROOT  = '/content/production_3d'
    CUBICASA_ROOT = '/content/CubiCasa5k'
    DATA_FOLDER   = '/content/data/cubicasa5k/'
    CKPT_DIR      = '/content/production_3d/checkpoints_paper'
else:
    PROJECT_ROOT  = '.'
    CUBICASA_ROOT = r'E:\JOB\CubiCasa5k'
    DATA_FOLDER   = r'C:/Users/kawayi_yaling/.cache/kagglehub/datasets/qmarva/cubicasa5k/versions/4/cubicasa5k/cubicasa5k/'
    CKPT_DIR      = './checkpoints_paper'

BEST_CKPT = os.path.join(CKPT_DIR, 'best_model.pth')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'e2e_test')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 把项目模块加入 path ──
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, CUBICASA_ROOT)
os.chdir(CUBICASA_ROOT)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── 前置检查 ──
checks = {
    'CubiCasa 数据目录': os.path.exists(DATA_FOLDER),
    'CubiCasa val.txt':  os.path.exists(os.path.join(DATA_FOLDER, 'val.txt')),
    '模型 checkpoint':   os.path.exists(BEST_CKPT),
    'project 模块目录':  os.path.exists(os.path.join(PROJECT_ROOT, 'model_arch.py')),
}
all_ok = True
for name, ok in checks.items():
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {name}')
    if not ok: all_ok = False

if not all_ok:
    print('\n⚠️  有路径不存在，请检查上方配置后重新运行')
else:
    print(f'\n✓ 环境就绪  device={DEVICE}')

  ✓  CubiCasa 数据目录
  ✓  CubiCasa val.txt
  ✗  模型 checkpoint
  ✗  project 模块目录

⚠️  有路径不存在，请检查上方配置后重新运行


## Cell 1 · 选图 & 加载 GT 标注

In [ ]:
from numpy import genfromtxt
from floortrans.loaders.house import House

# ── 选一张验证集图片 ──
# 改这个数字换图，0 是第一张
SAMPLE_IDX = 0

val_folders = genfromtxt(os.path.join(DATA_FOLDER, 'val.txt'), dtype='str')
FOLDER      = val_folders[SAMPLE_IDX].strip('/')
IMG_PATH    = os.path.join(DATA_FOLDER, FOLDER, 'F1_scaled.png')
SVG_PATH    = os.path.join(DATA_FOLDER, FOLDER, 'model.svg')

print(f'选中样本: {FOLDER}')
print(f'图片路径: {IMG_PATH}')

# ── 读取图片 ──
img_bgr = cv2.imread(IMG_PATH)
assert img_bgr is not None, f'图片读取失败: {IMG_PATH}'
IMG_RGB = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W    = IMG_RGB.shape[:2]
print(f'图片尺寸: {W} × {H}')

# ── 解析 SVG 获取 GT ──
house = House(SVG_PATH, H, W)
seg   = house.get_segmentation_tensor()   # (C, H, W)

WALL_CLASS_ID   = 2
DOOR_CLASS_ID   = 2
WINDOW_CLASS_ID = 1

GT_WALL_MASK = (seg[0] == WALL_CLASS_ID).astype(np.uint8)

# GT 门窗 bbox（从 seg[1] 提取）
gt_boxes, gt_labels = [], []
for cls_id, lbl in [(DOOR_CLASS_ID, 1), (WINDOW_CLASS_ID, 2)]:
    m    = (seg[1] == cls_id).astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts:
        if cv2.contourArea(cnt) < 100: continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        gt_boxes.append([x, y, x+bw, y+bh])
        gt_labels.append(lbl)
GT_BOXES  = np.array(gt_boxes,  dtype=np.float32) if gt_boxes  else np.zeros((0,4))
GT_LABELS = np.array(gt_labels, dtype=np.int64)   if gt_labels else np.zeros(0, dtype=np.int64)

print(f'GT wall 像素: {GT_WALL_MASK.sum()} / {H*W}  ({GT_WALL_MASK.mean()*100:.1f}%)')
print(f'GT 门窗数量: {len(GT_BOXES)}  (门={int((GT_LABELS==1).sum())}  窗={int((GT_LABELS==2).sum())})')

# ── 可视化 GT ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(IMG_RGB)
axes[0].set_title(f'原图  {W}×{H}'); axes[0].axis('off')
vis_gt = IMG_RGB.copy()
vis_gt[GT_WALL_MASK == 1] = (vis_gt[GT_WALL_MASK == 1] * 0.5 + np.array([0,220,0]) * 0.5).astype(np.uint8)
for b, l in zip(GT_BOXES, GT_LABELS):
    c = (0,0,255) if l==1 else (255,0,0)
    cv2.rectangle(vis_gt, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), c, 2)
axes[1].imshow(vis_gt)
axes[1].set_title(f'GT 标注  (绿=wall  蓝=door  红=window  {len(GT_BOXES)}个开口)')
axes[1].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '01_gt.png'), dpi=100)
plt.show()
print('✓ Cell 1 完成')

## Cell 2 · 自适应预处理

In [ ]:
from utils import detect_image_type, adaptive_preprocess, crop_to_wall_bbox

# ── 自动检测图片类型 ──
IMG_TYPE = detect_image_type(IMG_RGB)
print(f'检测到图片类型: {IMG_TYPE}')

# ── 自适应预处理 ──
IMG_PROCESSED = adaptive_preprocess(IMG_RGB, IMG_TYPE)
print(f'预处理前: mean={IMG_RGB.mean():.1f}  std={IMG_RGB.std():.1f}')
print(f'预处理后: mean={IMG_PROCESSED.mean():.1f}  std={IMG_PROCESSED.std():.1f}')

# ── 裁剪到 GT wall bbox（与训练保持一致）──
IMG_CROP, GT_CROP = crop_to_wall_bbox(
    IMG_PROCESSED, seg,
    wall_class_id=WALL_CLASS_ID,
    padding=10, min_size=64
)
if IMG_CROP is None:
    print('⚠️  crop 失败（无 wall 区域），使用原图继续')
    IMG_CROP = IMG_PROCESSED
    GT_CROP  = GT_WALL_MASK
else:
    GT_CROP = GT_CROP[0] == WALL_CLASS_ID  # 取 wall channel
    GT_CROP = GT_CROP.astype(np.uint8)

CH, CW = IMG_CROP.shape[:2]
print(f'裁剪后尺寸: {CW}×{CH}  (原图 {W}×{H})')

# ── 可视化 ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(IMG_RGB);        axes[0].set_title('原图');       axes[0].axis('off')
axes[1].imshow(IMG_PROCESSED);  axes[1].set_title(f'预处理 ({IMG_TYPE})'); axes[1].axis('off')
axes[2].imshow(IMG_CROP);       axes[2].set_title(f'裁剪到 wall bbox\n{CW}×{CH}'); axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_preprocess.png'), dpi=100)
plt.show()
print('✓ Cell 2 完成')

## Cell 3 · 模型推理（滑动窗口）

In [ ]:
from torchvision import transforms
from torchvision.ops import nms
from model_arch import build_model
from config import PaperConfig

# ── 加载模型 ──
print('加载模型...')
cfg   = PaperConfig()
model = build_model(cfg, DEVICE)
ckpt  = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'  ✓ checkpoint epoch={ckpt["epoch"]}  val_wall_iou={ckpt["val_wall_iou"]:.4f}')

# ── 推理参数 ──
TILE_SIZE    = 512
TILE_OVERLAP = 64
DET_THRESH   = 0.5
NORM_MEAN    = (0.485, 0.456, 0.406)
NORM_STD     = (0.229, 0.224, 0.225)

img_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

# ── 滑动窗口推理 ──
h, w  = IMG_CROP.shape[:2]
stride = TILE_SIZE - TILE_OVERLAP

wall_prob = np.zeros((h, w), dtype=np.float32)
wall_cnt  = np.zeros((h, w), dtype=np.float32)
all_boxes, all_scores, all_labels = [], [], []

ys = list(range(0, max(h - TILE_SIZE + 1, 1), stride))
xs = list(range(0, max(w - TILE_SIZE + 1, 1), stride))
if not ys or ys[-1] + TILE_SIZE < h: ys.append(max(h - TILE_SIZE, 0))
if not xs or xs[-1] + TILE_SIZE < w: xs.append(max(w - TILE_SIZE, 0))
tiles_total = len(ys) * len(xs)

t0 = time.time()
for ty in ys:
    for tx in xs:
        tile   = IMG_CROP[ty:ty+TILE_SIZE, tx:tx+TILE_SIZE].copy()
        th, tw = tile.shape[:2]
        if th < TILE_SIZE or tw < TILE_SIZE:
            tile = cv2.copyMakeBorder(tile, 0, TILE_SIZE-th, 0, TILE_SIZE-tw, cv2.BORDER_REFLECT_101)
        tile   = cv2.resize(tile, (TILE_SIZE, TILE_SIZE))
        tensor = img_tf(tile).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            out = model(tensor)

        prob = torch.softmax(out['seg_logits'], dim=1)[0, 1].cpu().numpy()
        prob = cv2.resize(prob, (tw, th), interpolation=cv2.INTER_LINEAR)
        wall_prob[ty:ty+th, tx:tx+tw] += prob[:th, :tw]
        wall_cnt[ty:ty+th,  tx:tx+tw] += 1

        det = out['det_outputs'][0]
        if len(det['boxes']) > 0:
            bxs  = det['boxes'].cpu().numpy()
            scrs = det['scores'].cpu().numpy()
            lbls = det['labels'].cpu().numpy()
            sx, sy = tw / TILE_SIZE, th / TILE_SIZE
            for b, s, l in zip(bxs, scrs, lbls):
                if s >= DET_THRESH:
                    all_boxes.append([b[0]*sx+tx, b[1]*sy+ty, b[2]*sx+tx, b[3]*sy+ty])
                    all_scores.append(float(s))
                    all_labels.append(int(l))

elapsed = time.time() - t0

# ── 合并结果 ──
wall_prob /= np.maximum(wall_cnt, 1)
PRED_WALL_MASK = (wall_prob > 0.5).astype(np.uint8)

if all_boxes:
    bt = torch.tensor(all_boxes,  dtype=torch.float32)
    st = torch.tensor(all_scores, dtype=torch.float32)
    lt = torch.tensor(all_labels, dtype=torch.int64)
    keep = nms(bt, st, iou_threshold=0.5)
    PRED_BOXES  = bt[keep].numpy()
    PRED_SCORES = st[keep].numpy()
    PRED_LABELS = lt[keep].numpy()
else:
    PRED_BOXES  = np.zeros((0,4), dtype=np.float32)
    PRED_SCORES = np.zeros(0,    dtype=np.float32)
    PRED_LABELS = np.zeros(0,    dtype=np.int64)

print(f'推理完成  tiles={tiles_total}  elapsed={elapsed:.1f}s')
print(f'pred wall 覆盖率: {PRED_WALL_MASK.mean()*100:.1f}%')
print(f'pred 检测框: {len(PRED_BOXES)}  (门={int((PRED_LABELS==1).sum())}  窗={int((PRED_LABELS==2).sum())})')

# ── 可视化 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(IMG_CROP); axes[0].set_title('输入（预处理后裁剪）'); axes[0].axis('off')

axes[1].imshow(wall_prob, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('wall probability map'); axes[1].axis('off')

vis = IMG_CROP.copy()
vis[PRED_WALL_MASK==1] = (vis[PRED_WALL_MASK==1]*0.5 + np.array([0,200,0])*0.5).astype(np.uint8)
for b, l, s in zip(PRED_BOXES, PRED_LABELS, PRED_SCORES):
    c = (0,0,255) if l==1 else (255,80,0)
    cv2.rectangle(vis, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), c, 2)
    cv2.putText(vis, f'{s:.2f}', (int(b[0]), max(int(b[1])-4,0)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)
axes[2].imshow(vis)
axes[2].set_title(f'预测结果  wall+{len(PRED_BOXES)}检测框')
axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_inference.png'), dpi=100)
plt.show()
print('✓ Cell 3 完成')

## Cell 4 · 分割质量验证（IoU mask）

In [ ]:
# ── GT 对齐到裁剪图尺寸 ──
gt_mask_aligned = cv2.resize(
    GT_CROP.astype(np.float32),
    (PRED_WALL_MASK.shape[1], PRED_WALL_MASK.shape[0]),
    interpolation=cv2.INTER_NEAREST
).astype(np.uint8)

# ── 计算 IoU mask ──
pred = PRED_WALL_MASK.astype(bool)
gt   = gt_mask_aligned.astype(bool)
tp   = (pred & gt).sum()
fp   = (pred & ~gt).sum()
fn   = (~pred & gt).sum()
IOU_MASK = tp / (tp + fp + fn + 1e-8)

# ── 差异图（TP/FP/FN 三色）──
diff = np.zeros((*gt.shape, 3), dtype=np.uint8)
diff[pred & gt]   = [255, 255,   0]   # 黄 = TP
diff[pred & ~gt]  = [255,   0,   0]   # 红 = FP
diff[~pred & gt]  = [  0,  80, 255]   # 蓝 = FN

TARGET = 0.75
flag   = '✓' if IOU_MASK >= TARGET else '✗'
print(f'IoU mask = {IOU_MASK:.4f}  (目标 > {TARGET})  {flag}')
print(f'  TP={tp}  FP={fp}  FN={fn}')
print(f'  Precision={tp/(tp+fp+1e-8):.3f}  Recall={tp/(tp+fn+1e-8):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(gt_mask_aligned, cmap='gray'); axes[0].set_title('GT wall mask'); axes[0].axis('off')
axes[1].imshow(PRED_WALL_MASK,  cmap='gray'); axes[1].set_title('Pred wall mask'); axes[1].axis('off')
axes[2].imshow(diff)
axes[2].set_title(f'差异图  IoU={IOU_MASK:.4f}\n黄=TP  红=FP  蓝=FN')
axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_seg_eval.png'), dpi=100)
plt.show()
print('✓ Cell 4 完成')

## Cell 5 · 矢量化（Section 2.4 完整链路）

In [ ]:
from vector_logic import (
    morphological_preprocessing,
    find_wall_angles,
    extract_wall_segments_at_angle,
    shrinking_algorithm,
    resolve_overlapping_boxes,
    vectorize_wall_mask,
)
from postprocess_config import VectorizationConfig

VECT_CFG = VectorizationConfig()   # iou_threshold=0.85, min_segment_area=200

print('Step 1: 形态学预处理...')
mask_cleaned = morphological_preprocessing(PRED_WALL_MASK, VECT_CFG)
print(f'  像素: {PRED_WALL_MASK.sum()} → {mask_cleaned.sum()}')

print('Step 2: Hough 角度检测...')
angles = find_wall_angles(mask_cleaned, VECT_CFG)
print(f'  检测到角度: {[round(a,1) for a in angles]}')

print('Step 3-4: 按角度提取轮廓...')
all_contours = []
for a in angles:
    cnts = extract_wall_segments_at_angle(mask_cleaned, a, VECT_CFG)
    all_contours.extend(cnts)
print(f'  提取到轮廓: {len(all_contours)}')

print('Step 5: Shrinking 算法...')
raw_boxes = []
for cnt in all_contours:
    box = shrinking_algorithm(mask_cleaned, cnt, VECT_CFG)
    if box is not None:
        raw_boxes.append(box)
print(f'  Shrinking 后: {len(raw_boxes)} bbox')

print('Step 6: 重叠消解...')
WALL_BOXES = resolve_overlapping_boxes(raw_boxes, VECT_CFG)
print(f'  消解后: {len(WALL_BOXES)} bbox')

# ── 可视化矢量化过程 ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 原始 mask
axes[0].imshow(PRED_WALL_MASK, cmap='gray')
axes[0].set_title('原始预测 mask'); axes[0].axis('off')

# 修补后 mask + 轮廓
vis_mask = cv2.cvtColor(mask_cleaned * 255, cv2.COLOR_GRAY2RGB)
cv2.drawContours(vis_mask, all_contours, -1, (255,100,0), 2)
axes[1].imshow(vis_mask)
axes[1].set_title(f'修补后 + {len(all_contours)} 轮廓 (橙色)')
axes[1].axis('off')

# 最终 wall bbox
vis_vec = IMG_CROP.copy()
for b in WALL_BOXES:
    x1,y1,x2,y2 = [int(v) for v in b]
    cv2.rectangle(vis_vec, (x1,y1), (x2,y2), (0,255,80), 2)
for b, l in zip(PRED_BOXES, PRED_LABELS):
    c = (0,0,255) if l==1 else (255,80,0)
    cv2.rectangle(vis_vec, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), c, 2)
axes[2].imshow(vis_vec)
axes[2].set_title(f'矢量化结果: {len(WALL_BOXES)} 墙  {len(PRED_BOXES)} 开口')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_vectorize.png'), dpi=100)
plt.show()
print('✓ Cell 5 完成')

## Cell 6 · 矢量化质量验证（IoU vect）

In [ ]:
from vector_logic import compute_vectorization_iou, wall_boxes_to_mask

# ── 把 wall_boxes 转回 binary mask 和 GT 比较 ──
vect_mask = wall_boxes_to_mask(WALL_BOXES, gt_mask_aligned.shape)
IOU_VECT  = compute_vectorization_iou(WALL_BOXES, gt_mask_aligned)

# ── 矢量化保真度：pred_mask 有多少被 bbox 覆盖 ──
if PRED_WALL_MASK.sum() > 0:
    vect_coverage = float((vect_mask & PRED_WALL_MASK.astype(bool)).sum()) / PRED_WALL_MASK.sum()
else:
    vect_coverage = 0.0

TARGET_VECT = 0.60
flag = '✓' if IOU_VECT >= TARGET_VECT else '✗'
print(f'IoU vect     = {IOU_VECT:.4f}  (目标 > {TARGET_VECT})  {flag}')
print(f'IoU mask     = {IOU_MASK:.4f}')
print(f'保真度        = {vect_coverage:.3f}  (矢量化后保留了多少 pred 像素)')
print(f'wall bbox 数 = {len(WALL_BOXES)}')

if IOU_VECT < IOU_MASK * 0.8:
    print('⚠️  矢量化损失较大，考虑降低 iou_threshold 或 min_segment_area')

# ── 可视化 ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(gt_mask_aligned, cmap='gray');  axes[0].set_title('GT mask'); axes[0].axis('off')
axes[1].imshow(PRED_WALL_MASK,  cmap='gray');  axes[1].set_title(f'Pred mask  IoU={IOU_MASK:.3f}'); axes[1].axis('off')
axes[2].imshow(vect_mask,       cmap='gray');  axes[2].set_title(f'Vect mask  IoU={IOU_VECT:.3f}'); axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_vect_eval.png'), dpi=100)
plt.show()
print('✓ Cell 6 完成')

## Cell 7 · 门窗检测质量（F1）

In [ ]:
from torchvision.ops import box_iou as tv_box_iou

def det_metrics(pred_boxes, pred_scores, pred_labels,
                gt_boxes, gt_labels,
                iou_thresh=0.5, score_thresh=0.5):
    keep = pred_scores >= score_thresh
    pb, pl = pred_boxes[keep], pred_labels[keep]
    tp = fp = fn = 0
    if len(gt_boxes) == 0:
        return len(pb), 0, 0
    if len(pb) == 0:
        return 0, 0, len(gt_boxes)
    iou_mat = tv_box_iou(
        torch.tensor(pb,       dtype=torch.float32),
        torch.tensor(gt_boxes, dtype=torch.float32)
    ).numpy()
    matched = set()
    for i in range(len(pb)):
        j = iou_mat[i].argmax()
        if iou_mat[i, j] >= iou_thresh and j not in matched:
            tp += 1; matched.add(j)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched)
    return tp, fp, fn

# GT bbox 可能在原图坐标系，需要对齐到裁剪图
# 这里用一个简化策略：直接用 GT_BOXES（已在 GT 尺寸下），
# PRED_BOXES 在裁剪图尺寸下，两者如果尺寸相同则直接比较
# 如果尺寸不同则按比例缩放 GT_BOXES
scale_x = IMG_CROP.shape[1] / W
scale_y = IMG_CROP.shape[0] / H
gt_boxes_scaled = GT_BOXES.copy()
if len(gt_boxes_scaled) > 0:
    gt_boxes_scaled[:, [0,2]] *= scale_x
    gt_boxes_scaled[:, [1,3]] *= scale_y

tp, fp, fn = det_metrics(
    PRED_BOXES, PRED_SCORES, PRED_LABELS,
    gt_boxes_scaled, GT_LABELS
)
precision = tp / (tp + fp + 1e-8)
recall    = tp / (tp + fn + 1e-8)
DET_F1    = 2 * precision * recall / (precision + recall + 1e-8)

TARGET_F1 = 0.60
flag = '✓' if DET_F1 >= TARGET_F1 else '✗'
print(f'检测 F1    = {DET_F1:.4f}  (目标 > {TARGET_F1})  {flag}')
print(f'  Precision= {precision:.3f}  Recall= {recall:.3f}')
print(f'  TP={tp}  FP={fp}  FN={fn}')
print(f'  GT 总数={len(gt_boxes_scaled)}  Pred 总数={len(PRED_BOXES)}')

# ── 门窗落墙匹配率 ──
door_m = door_t = win_m = win_t = 0
ch2, cw2 = PRED_WALL_MASK.shape
MATCH_OVERLAP_THRESH = 0.3
for b, l in zip(PRED_BOXES, PRED_LABELS):
    x1=max(0,int(b[0])); y1=max(0,int(b[1]))
    x2=min(cw2,int(b[2])); y2=min(ch2,int(b[3]))
    if x2<=x1 or y2<=y1: continue
    area = (x2-x1)*(y2-y1)
    ov   = PRED_WALL_MASK[y1:y2,x1:x2].sum() / (area+1e-8)
    ok   = ov >= MATCH_OVERLAP_THRESH
    if   int(l)==1: door_t+=1; door_m+=int(ok)
    elif int(l)==2: win_t+=1;  win_m+=int(ok)
DOOR_MATCH = door_m/(door_t+1e-8)
WIN_MATCH  = win_m/(win_t+1e-8)
print(f'  门落墙率 = {DOOR_MATCH:.3f} ({door_m}/{door_t})')
print(f'  窗落墙率 = {WIN_MATCH:.3f} ({win_m}/{win_t})')

# ── 可视化 GT vs Pred 对比 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

vis_gt2 = IMG_CROP.copy()
for b, l in zip(gt_boxes_scaled, GT_LABELS):
    c = (0,0,200) if l==1 else (200,0,0)
    cv2.rectangle(vis_gt2, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), c, 2)
axes[0].imshow(vis_gt2)
axes[0].set_title(f'GT  ({len(gt_boxes_scaled)} 个开口)')
axes[0].axis('off')

vis_pred2 = IMG_CROP.copy()
for b, l, s in zip(PRED_BOXES, PRED_LABELS, PRED_SCORES):
    c = (0,0,200) if l==1 else (200,0,0)
    cv2.rectangle(vis_pred2, (int(b[0]),int(b[1])), (int(b[2]),int(b[3])), c, 2)
    cv2.putText(vis_pred2, f'{s:.2f}', (int(b[0]), max(int(b[1])-4,10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)
axes[1].imshow(vis_pred2)
axes[1].set_title(f'Pred  F1={DET_F1:.3f}  ({len(PRED_BOXES)} 个开口)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_det_eval.png'), dpi=100)
plt.show()
print('✓ Cell 7 完成')

## Cell 8 · 3D 重建

In [ ]:
from reconstruct_3d import match_openings_to_walls, build_3d_model

# ── 门窗匹配到墙体 ──
openings = match_openings_to_walls(WALL_BOXES, PRED_BOXES, PRED_LABELS)
doors    = [o for o in openings if o['type'] == 'door']
windows  = [o for o in openings if o['type'] == 'window']
print(f'门窗匹配: {len(openings)} 个 (门={len(doors)}  窗={len(windows)})')

# 打印匹配详情
for op in openings[:5]:   # 只打印前5个
    print(f'  {op["type"]:6s} → wall_{op["wall_idx"]}  '
          f'opening={op["opening_box"]}  wall={op["wall_box"]}')
if len(openings) > 5:
    print(f'  ... 共 {len(openings)} 个')

# ── 3D 重建 ──
GLB_PATH = os.path.join(OUTPUT_DIR, f'{FOLDER.replace("/","_")}_3d.glb')
t0 = time.time()
scene = build_3d_model(
    wall_boxes  = WALL_BOXES,
    openings    = openings,
    output_path = GLB_PATH,
)
elapsed_3d = time.time() - t0

if scene is not None:
    print(f'\n✓ 3D 重建完成  elapsed={elapsed_3d:.1f}s')
    print(f'  GLB 路径: {GLB_PATH}')
    print(f'  文件大小: {os.path.getsize(GLB_PATH)/1024:.1f} KB')
    # 场景统计
    geom_names = list(scene.geometry.keys()) if hasattr(scene, 'geometry') else []
    print(f'  场景对象: {geom_names[:8]}{'...' if len(geom_names)>8 else ''}')
else:
    print('⚠️  3D 重建失败，请检查 trimesh 是否安装')
    print('   pip install trimesh')

print('✓ Cell 8 完成')

## Cell 9 · 3D 几何合理性检查

In [ ]:
# ── 几何合理性：bbox 宽高比检查 ──
aspect_ok = aspect_bad = 0
bad_boxes = []
for b in WALL_BOXES:
    w_b = b[2] - b[0]
    h_b = b[3] - b[1]
    if h_b < 1: continue
    ratio = w_b / h_b
    if 0.1 < ratio < 10:
        aspect_ok += 1
    else:
        aspect_bad += 1
        bad_boxes.append((b, ratio))

GEOM_OK_RATE = aspect_ok / (len(WALL_BOXES) + 1e-8)
flag = '✓' if GEOM_OK_RATE >= 0.9 else '✗'
print(f'3D 几何合理性 = {GEOM_OK_RATE:.3f}  ({aspect_ok}/{len(WALL_BOXES)} 个 bbox 比例正常)  {flag}')
if bad_boxes:
    print(f'  异常 bbox: {len(bad_boxes)} 个')
    for b, r in bad_boxes[:3]:
        print(f'    box={[int(v) for v in b]}  宽高比={r:.2f}')

# ── 门窗尺寸合理性 ──
from postprocess_config import PIXELS_PER_METER, DOOR_HEIGHT, WINDOW_HEIGHT
door_size_ok = win_size_ok = 0
for op in openings:
    b = op['opening_box']
    w_b = abs(b[2]-b[0]); h_b = abs(b[3]-b[1])
    longer = max(w_b, h_b) / PIXELS_PER_METER   # 较长边对应现实尺寸（米）
    if op['type'] == 'door':
        ok = 0.6 < longer < 3.5   # 门宽 0.6~3.5m 合理
        door_size_ok += int(ok)
    else:
        ok = 0.3 < longer < 4.0   # 窗宽 0.3~4m 合理
        win_size_ok += int(ok)

print(f'门尺寸合理: {door_size_ok}/{len(doors)}')
print(f'窗尺寸合理: {win_size_ok}/{len(windows)}')

# ── 打印 3D 参数 ──
print(f'\n3D 建模参数:')
print(f'  pixels/meter   = {PIXELS_PER_METER}')
print(f'  wall_height    = {2.8} m')
print(f'  door_height    = {DOOR_HEIGHT} m')
print(f'  window_height  = {WINDOW_HEIGHT} m')
print(f'  墙体总数       = {len(WALL_BOXES)}')
print(f'  开口总数       = {len(openings)}')

print('✓ Cell 9 完成')

## Cell 10 · Staging + 审核流程模拟

In [ ]:
import uuid, shutil
from persistence_service import approve, reject, list_pending, _append_audit_log
from postprocess_config import STAGING_CFG

# ── 构造一个 task 元数据（模拟 preview_service 的输出）──
TASK_ID      = str(uuid.uuid4())[:8]
staging_task = os.path.join(STAGING_CFG.staging_dir, TASK_ID)
os.makedirs(staging_task, exist_ok=True)

# 把 GLB 和结果 JSON 复制到 staging
if os.path.exists(GLB_PATH):
    shutil.copy(GLB_PATH, os.path.join(staging_task, f'{TASK_ID}_3d.glb'))

task_meta = {
    'task_id':    TASK_ID,
    'status':     'pending',
    'image_path': IMG_PATH,
    'glb_path':   os.path.join(staging_task, f'{TASK_ID}_3d.glb'),
    'stats': {
        'n_walls':   len(WALL_BOXES),
        'n_doors':   len(doors),
        'n_windows': len(windows),
        'iou_mask':  round(float(IOU_MASK), 4),
        'iou_vect':  round(float(IOU_VECT), 4),
        'det_f1':    round(float(DET_F1), 4),
    },
    'elapsed': 0.0,
}
with open(os.path.join(staging_task, 'meta.json'), 'w') as f:
    json.dump(task_meta, f, indent=2)
print(f'Staging 任务已创建: {TASK_ID}')
print(f'  路径: {staging_task}')

# ── 查询待审核队列 ──
pending = list_pending()
print(f'\n当前待审核任务数: {len(pending)}')
for p in pending[-3:]:   # 显示最新3个
    print(f'  {p.get("task_id")}  walls={p.get("stats",{}).get("n_walls")}  iou={p.get("stats",{}).get("iou_mask")}')

# ── 根据指标自动决策 ──
AUTO_APPROVE_THRESH = 0.72
AUTO_REJECT_THRESH  = 0.40

if IOU_MASK >= AUTO_APPROVE_THRESH and DET_F1 >= 0.50:
    print(f'\n自动决策: ✅ AUTO_APPROVE  (IoU={IOU_MASK:.3f} ≥ {AUTO_APPROVE_THRESH})')
    result = approve(TASK_ID, upload_to_gcs=False)
elif IOU_MASK <= AUTO_REJECT_THRESH:
    print(f'\n自动决策: ❌ AUTO_REJECT  (IoU={IOU_MASK:.3f} ≤ {AUTO_REJECT_THRESH})')
    result = reject(TASK_ID, reason='wall_missing', note='IoU 低于自动拒绝阈值', keep_file=True)
else:
    print(f'\n自动决策: 🔍 HUMAN_REVIEW  ({AUTO_REJECT_THRESH} < IoU={IOU_MASK:.3f} < {AUTO_APPROVE_THRESH})')
    review_mark = {
        'status':     'human_review',
        'iou_mask':   float(IOU_MASK),
        'iou_vect':   float(IOU_VECT),
        'det_f1':     float(DET_F1),
        'timestamp':  time.strftime('%Y-%m-%dT%H:%M:%SZ'),
    }
    with open(os.path.join(staging_task, 'human_review.json'), 'w') as f:
        json.dump(review_mark, f, indent=2)
    result = {'task_id': TASK_ID, 'status': 'human_review'}

print(f'  结果: {result}')
print('\n（工程化后这里会真正调用 persistence_service，触发 GCS 归档或拒绝标注）')
print('✓ Cell 10 完成')

## Cell 11 · 全流程指标汇总

In [ ]:
from postprocess_config import ValidationTargets

TARGETS = ValidationTargets()

results = {
    'IoU mask':       (float(IOU_MASK),  TARGETS.iou_mask_min),
    'IoU vect':       (float(IOU_VECT),  TARGETS.iou_vect_min),
    '检测 F1':        (float(DET_F1),    TARGETS.det_f1_min),
    '门落墙率':        (float(DOOR_MATCH), TARGETS.match_rate_min),
    '窗落墙率':        (float(WIN_MATCH),  TARGETS.match_rate_min),
    '3D几何合理性':    (float(GEOM_OK_RATE), TARGETS.geom_ok_min),
}
pipeline_ok = [
    ('数据加载',    True),
    ('预处理',      True),
    ('推理',        PRED_WALL_MASK.sum() > 0),
    ('矢量化',      len(WALL_BOXES) > 0),
    ('3D重建',      scene is not None),
    ('Staging',     os.path.exists(staging_task)),
]

print('=' * 55)
print('端到端验证报告')
print('=' * 55)
print(f'样本: {FOLDER}')
print(f'图片: {W}×{H}')
print()

print('── 流水线完整性 ──')
all_pipeline_ok = True
for step, ok in pipeline_ok:
    sym = '✓' if ok else '✗'
    print(f'  {sym}  {step}')
    if not ok: all_pipeline_ok = False

print()
print('── 质量指标 ──')
print(f'  {"指标":<15} {"值":>8}   {"目标":>8}   {"通过"}')
print('  ' + '-'*45)
all_metrics_ok = True
for name, (val, target) in results.items():
    ok  = val >= target
    sym = '✓' if ok else '✗'
    print(f'  {name:<15} {val:>8.4f}   {target:>8.2f}   {sym}')
    if not ok: all_metrics_ok = False

print()
print('── 产出文件 ──')
outputs = [
    ('GLB 3D 模型',   GLB_PATH),
    ('Staging 目录',  staging_task),
    ('可视化图片',    OUTPUT_DIR),
]
for name, path in outputs:
    exists = os.path.exists(path)
    print(f'  {"✓" if exists else "✗"}  {name}: {path}')

print()
print('=' * 55)
if all_pipeline_ok:
    print('✅  流水线全部步骤跑通，无断点')
else:
    print('⚠️  流水线有断点，请检查 ✗ 步骤')

if all_metrics_ok:
    print('✅  所有指标达到目标阈值')
else:
    passing = sum(1 for v,t in results.values() if v>=t)
    print(f'⚠️  {passing}/{len(results)} 个指标达标，查看上方明细')
print('=' * 55)

# 保存 JSON 报告
report = {
    'folder':   FOLDER,
    'img_size': [W, H],
    'metrics':  {k: {'value': v, 'target': t, 'pass': v>=t} for k,(v,t) in results.items()},
    'pipeline': {s: ok for s,ok in pipeline_ok},
    'n_walls':  len(WALL_BOXES),
    'n_doors':  len(doors),
    'n_windows':len(windows),
    'glb_path': GLB_PATH,
    'task_id':  TASK_ID,
}
report_path = os.path.join(OUTPUT_DIR, 'e2e_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f'\n报告已保存: {report_path}')